# Day 6 – Baseline Modeling

**Objective:** Train a baseline LightGBM regression model on the engineered M5 sales-forecasting features, evaluate it on a time-based validation split, then use feature importance to select a leaner feature set and compare performance.

**Workflow:** Explore feature chunks → time-based train/validation split → baseline model → performance evaluation → feature importance analysis → feature selection → retrain & compare → key findings.


## 1. Import Libraries & Explore Feature-Engineered Data

Inspect the Parquet feature chunks produced in the previous stage (Day 5): check shape, columns, target distribution, and missing values on a sample chunk, then confirm the date coverage and file listing.


In [1]:
import os
import pandas as pd

path = r"../data/processed/feature_chunks"

files = sorted([
    f for f in os.listdir(path)
    if f.endswith(".parquet")
])

print("Number of chunks:", len(files))

sample = pd.read_parquet(os.path.join(path, files[0]))

print("\nShape:")
print(sample.shape)

print("\nColumns:")
print(sample.columns.tolist())

print("\nTarget column statistics:")
print(sample["sales"].describe())

print("\nMissing values:")
print(sample.isnull().sum().sort_values(ascending=False).head(20))


Number of chunks: 11

Shape:
(5655000, 38)

Columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'day', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_std_14', 'rolling_mean_28', 'rolling_std_28', 'is_weekend', 'price_lag_1', 'price_diff_1', 'price_pct_change', 'price_max', 'price_relative_to_max']

Target column statistics:
count    5.655000e+06
mean     1.304095e+00
std      3.989725e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      5.990000e+02
Name: sales, dtype: float64

Missing values:
id                  0
rolling_mean_14     0
sell_price          0
lag_1               0
lag_7               0
lag_14              0
lag_28              0
rolling_mean_7      0
rolling_std_7       0
rolli

In [2]:
import pandas as pd
import os

path = r"../data/processed/feature_chunks"

files = sorted([
    f for f in os.listdir(path)
    if f.endswith(".parquet")
])

df = pd.read_parquet(
    os.path.join(path, files[0]),
    columns=["date"]
)

print("Min Date:", df["date"].min())
print("Max Date:", df["date"].max())
print("Unique Dates:", df["date"].nunique())

Min Date: 2011-02-26 00:00:00
Max Date: 2016-04-24 00:00:00
Unique Dates: 1885


In [4]:
import os

path = r"../data/processed/feature_chunks"

files = sorted(os.listdir(path))

print("Number of files:", len(files))

for f in files[:20]:
    print(f)
    

Number of files: 11
features_chunk_0001.parquet
features_chunk_0002.parquet
features_chunk_0003.parquet
features_chunk_0004.parquet
features_chunk_0005.parquet
features_chunk_0006.parquet
features_chunk_0007.parquet
features_chunk_0008.parquet
features_chunk_0009.parquet
features_chunk_0010.parquet
features_chunk_0011.parquet


## 2. Load Feature Engineered Chunks

Scan across **all** feature chunks (not just the first) to determine the complete date range of the dataset. This full range is required to define chronological validation/test boundaries in the next step.


In [6]:
import pandas as pd
import os

path = r"../data/processed/feature_chunks"

files = sorted([
    f for f in os.listdir(path)
    if f.endswith(".parquet")
])

all_dates = set()

for file in files:
    df_dates = pd.read_parquet(
        os.path.join(path, file),
        columns=["date"]
    )

    all_dates.update(df_dates["date"].unique())

dates = sorted(all_dates)

print("Total unique dates:", len(dates))
print("Min Date:", dates[0])
print("Max Date:", dates[-1])

valid_start = dates[-56]
test_start = dates[-28]

print("\nValidation Start:", valid_start)
print("Test Start:", test_start)

Total unique dates: 1885
Min Date: 2011-02-26 00:00:00
Max Date: 2016-04-24 00:00:00

Validation Start: 2016-02-29 00:00:00
Test Start: 2016-03-28 00:00:00


In [8]:
import pandas as pd
import os

path = r"../data/processed/feature_chunks"

for file in [
    "features_chunk_0001.parquet",
    "features_chunk_0006.parquet",
    "features_chunk_0011.parquet",
    "features_chunk_0007.parquet"
]:
    df = pd.read_parquet(
        os.path.join(path, file),
        columns=["date"]
    )

    print(file)
    print(df["date"].min())
    print(df["date"].max())
    print("-"*50)

features_chunk_0001.parquet
2011-02-26 00:00:00
2016-04-24 00:00:00
--------------------------------------------------
features_chunk_0006.parquet
2011-02-26 00:00:00
2016-04-24 00:00:00
--------------------------------------------------
features_chunk_0011.parquet
2011-02-26 00:00:00
2016-04-24 00:00:00
--------------------------------------------------
features_chunk_0007.parquet
2011-02-26 00:00:00
2016-04-24 00:00:00
--------------------------------------------------


## 3. Define Time-Based Train/Validation Split

This is time-series sales data, so a **chronological split** is used instead of random sampling to prevent data leakage. The last 56 days are held out for validation + test, with the final 28 days reserved for test (validation starts `2016-02-29`, test starts `2016-03-28`).


In [9]:
path = r"../data/processed/feature_chunks"

files = sorted([
    f for f in os.listdir(path)
    if f.endswith(".parquet")
])[:4]

In [10]:
VALID_START = pd.Timestamp("2016-02-29")
TEST_START = pd.Timestamp("2016-03-28")

## 4. Define Features and Target

Specify the target column, columns to exclude from the model (identifiers and raw date), and the categorical columns that will be handled natively by LightGBM.


In [11]:
TARGET = "sales"

DROP_COLS = [
    "sales",
    "id",
    "day",
    "date"
]

CATEGORICAL_COLS = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "weekday",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]


### Build Train/Validation Sets from Chunks

Iterate over each feature chunk: cast categorical columns, split rows chronologically into training and validation sets, then concatenate all chunks into the final training and validation DataFrames.


In [12]:
train_parts = []
valid_parts = []


In [13]:
for file in files:

    print(f"Processing {file}")

    df = pd.read_parquet(
        os.path.join(path, file)
    )

    # categorical conversion
    for col in CATEGORICAL_COLS:
        df[col] = df[col].astype("category")

    # time split
    train_df = df[df["date"] < VALID_START]
    valid_df = df[
        (df["date"] >= VALID_START) &
        (df["date"] < TEST_START)
    ]

    train_parts.append(train_df)
    valid_parts.append(valid_df)

    print(
        f"Train: {train_df.shape} | "
        f"Valid: {valid_df.shape}"
    )

Processing features_chunk_0001.parquet
Train: (5487000, 38) | Valid: (84000, 38)
Processing features_chunk_0002.parquet
Train: (5487000, 38) | Valid: (84000, 38)
Processing features_chunk_0003.parquet
Train: (5487000, 38) | Valid: (84000, 38)
Processing features_chunk_0004.parquet
Train: (5487000, 38) | Valid: (84000, 38)


In [14]:
train_df = pd.concat(
    train_parts,
    ignore_index=True
)

valid_df = pd.concat(
    valid_parts,
    ignore_index=True
)

print("\nFinal Shapes")
print("Train:", train_df.shape)
print("Validation:", valid_df.shape)


Final Shapes
Train: (21948000, 38)
Validation: (336000, 38)


Preview the resulting feature/target matrices before finalizing categorical dtypes on the full concatenated data.


In [15]:
# =========================
# Features & Target
# =========================

TARGET = "sales"

DROP_COLS = [
    "sales",
    "id",
    "day",
    "date"
]

X_train = train_df.drop(columns=DROP_COLS)
y_train = train_df[TARGET]

X_valid = valid_df.drop(columns=DROP_COLS)
y_valid = valid_df[TARGET]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_valid:", X_valid.shape)
print("y_valid:", y_valid.shape)

X_train: (21948000, 34)
y_train: (21948000,)
X_valid: (336000, 34)
y_valid: (336000,)


## 5. Prepare Categorical Features

Re-apply categorical dtype casting on the concatenated train/validation DataFrames (dtype can be lost during `pd.concat`), then finalize the `X`/`y` matrices that will be passed to the model.


In [16]:
CATEGORICAL_COLS = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "weekday",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

In [17]:
# Re-cast categorical dtypes directly on the concatenated DataFrames
# (ensures dtype consistency after pd.concat, which can reset category dtype)
for col in CATEGORICAL_COLS:
    train_df[col] = train_df[col].astype("category")
    valid_df[col] = valid_df[col].astype("category")

In [18]:
# Finalize feature/target matrices used for training (supersedes the earlier preview)
TARGET = "sales"

DROP_COLS = [
    "sales",
    "id",
    "day",
    "date"
]

X_train = train_df.drop(columns=DROP_COLS)
y_train = train_df[TARGET]

X_valid = valid_df.drop(columns=DROP_COLS)
y_valid = valid_df[TARGET]

In [19]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_valid:", X_valid.shape)
print("y_valid:", y_valid.shape)

X_train: (21948000, 34)
y_train: (21948000,)
X_valid: (336000, 34)
y_valid: (336000,)


In [20]:
print(X_train.dtypes.value_counts())

float32     16
int8         6
int16        2
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64


## 6. Train Baseline LightGBM Model

Train a baseline LightGBM regressor with standard hyperparameters (300 trees, learning rate 0.05, subsampling and column sampling at 0.8) on the full feature set, using native categorical support and an evaluation set for monitoring.


In [21]:
from lightgbm import LGBMRegressor

model = LGBMRegressor(
    objective="regression",
    metric="rmse",

    n_estimators=300,
    learning_rate=0.05,

    num_leaves=64,
    max_depth=-1,

    subsample=0.8,
    colsample_bytree=0.8,

    random_state=42,
    n_jobs=-1,
    verbose=-1
)

In [22]:
model.fit(
    X_train,
    y_train,

    eval_set=[(X_valid, y_valid)],

    categorical_feature=CATEGORICAL_COLS
)

LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, metric='rmse',
              n_estimators=300, n_jobs=-1, num_leaves=64,
              objective='regression', random_state=42, subsample=0.8,
              verbose=-1)

## 7. Evaluate Model Performance

Generate predictions on the validation set and compute RMSE and MAE to establish the baseline performance benchmark.


In [23]:
preds = model.predict(X_valid)

In [24]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_valid, preds))
mae = mean_absolute_error(y_valid, preds)

print("RMSE:", rmse)
print("MAE :", mae)

RMSE: 2.016737402307274
MAE : 1.019657175635555


## 8. Analyze Feature Importance

Rank all features by LightGBM importance score (highest and lowest) to identify which predictors drive the model and which contribute little to no predictive value.


In [25]:
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="importance",
    ascending=False
)

print(importance_df.head(20))

                  feature  importance
0                 item_id        2401
18                  lag_1        2263
5                wm_yr_wk        1416
22         rolling_mean_7        1324
20                 lag_14        1259
19                  lag_7        1190
21                 lag_28        1148
24        rolling_mean_14         832
7                    wday         771
26        rolling_mean_28         763
6                 weekday         696
8                   month         687
23          rolling_std_7         630
10           event_name_1         609
3                store_id         401
33  price_relative_to_max         373
14                snap_CA         321
17             sell_price         301
25         rolling_std_14         212
27         rolling_std_28         173


In [26]:
importance_df_sorted = importance_df.sort_values(
    by="importance",
    ascending=True
)

print(importance_df_sorted)

                  feature  importance
4                state_id           0
12           event_name_2           0
13           event_type_2           1
11           event_type_1           7
28             is_weekend          15
2                  cat_id          52
9                    year          74
30           price_diff_1         110
15                snap_TX         132
1                 dept_id         133
31       price_pct_change         134
16                snap_WI         135
32              price_max         167
29            price_lag_1         170
27         rolling_std_28         173
25         rolling_std_14         212
17             sell_price         301
14                snap_CA         321
33  price_relative_to_max         373
3                store_id         401
10           event_name_1         609
23          rolling_std_7         630
8                   month         687
6                 weekday         696
26        rolling_mean_28         763
7           

## 9. Perform Feature Selection

Drop the near-zero-importance features identified above (`state_id`, `event_name_2`, `event_type_2`) and rebuild the feature/target matrices and categorical column list for a leaner model.


In [27]:
# Initial manual attempt at dropping low-importance columns
# (superseded by the LOW_IMPORTANCE_COLS-based version in the next cell)
DROP_COLS_V2 = [
    "sales",
    "id",
    "day",
    "date",
    "state_id",
    "event_name_2",
    "event_type_2"
]

X_train_v2 = train_df.drop(columns=DROP_COLS_V2)
X_valid_v2 = valid_df.drop(columns=DROP_COLS_V2)

cat_cols_v2 = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "weekday",
    "event_name_1",
    "event_type_1"
]

In [28]:
# Final feature selection: drop the near-zero-importance columns identified above
LOW_IMPORTANCE_COLS = [
    "state_id",
    "event_name_2",
    "event_type_2"
]

DROP_COLS_V2 = [
    "sales",
    "id",
    "day",
    "date"
] + LOW_IMPORTANCE_COLS

X_train_v2 = train_df.drop(columns=DROP_COLS_V2)
y_train_v2 = train_df["sales"]

X_valid_v2 = valid_df.drop(columns=DROP_COLS_V2)
y_valid_v2 = valid_df["sales"]

In [29]:
CATEGORICAL_COLS_V2 = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "weekday",
    "event_name_1",
    "event_type_1"
]

## 10. Retrain Model After Feature Selection

Train a second LightGBM model (`model_v2`) with the same hyperparameters on the reduced feature set, to test whether removing low-importance features maintains or improves performance.


In [30]:
from lightgbm import LGBMRegressor

model_v2 = LGBMRegressor(
    objective="regression",
    metric="rmse",

    n_estimators=300,
    learning_rate=0.05,

    num_leaves=64,
    max_depth=-1,

    subsample=0.8,
    colsample_bytree=0.8,

    random_state=42,
    n_jobs=-1,
    verbose=-1
)

model_v2.fit(
    X_train_v2,
    y_train_v2,

    eval_set=[(X_valid_v2, y_valid_v2)],
    categorical_feature=CATEGORICAL_COLS_V2
)

LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, metric='rmse',
              n_estimators=300, n_jobs=-1, num_leaves=64,
              objective='regression', random_state=42, subsample=0.8,
              verbose=-1)

In [31]:
preds_v2 = model_v2.predict(X_valid_v2)

## 11. Compare Baseline vs Optimized Model

Evaluate the reduced-feature model on the validation set and compare its RMSE/MAE against the baseline model from Section 7.


In [32]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import numpy as np

rmse_v2 = np.sqrt(
    mean_squared_error(
        y_valid_v2,
        preds_v2
    )
)

mae_v2 = mean_absolute_error(
    y_valid_v2,
    preds_v2
)

print("=" * 40)
print("Baseline Model Results")
print("=" * 40)

print(f"RMSE: {rmse_v2:.6f}")
print(f"MAE : {mae_v2:.6f}")

Baseline Model Results
RMSE: 2.010984
MAE : 1.019113


## 12. Key Findings and Business Insights

- **Baseline model** (34 features): RMSE ≈ 2.017, MAE ≈ 1.020 on the validation set.
- **Optimized model** (after dropping `state_id`, `event_name_2`, `event_type_2`): RMSE ≈ 2.011, MAE ≈ 1.019 — essentially matching baseline accuracy with a smaller, simpler feature set.
- **Top predictive signals:** `item_id`, `lag_1` (previous day's sales), `wm_yr_wk` (week identifier), and the rolling mean/std features (7/14/28-day windows) dominate feature importance — consistent with strong autocorrelation in daily retail sales.
- **Lowest-value features:** `state_id`, `event_name_2`, and `event_type_2` contributed virtually no predictive signal and were safely removed without hurting performance.
- **Takeaway:** A small set of lag and rolling-window features captures most of the predictive power for this baseline; further gains will likely come from richer lag/rolling engineering or hyperparameter tuning rather than adding more raw categorical columns.
